# Armonización a Dimensiones Conformes — Corrección Entrega I

```
================================================================================
ENTREGA 1 (CORRECCION v2) - ARMONIZACION A DIMENSIONES CONFORMES
================================================================================
Responde a la retroalimentacion del tutor: construye la capa conforme que
permite filtros globales sobre las 12 fuentes sin ID de persona compartido.

Para cada dataset:
  1. Lee el CSV original.
  2. Mapea sus atributos nativos a las 4 dimensiones conformes:
       - dim_genero          (M / F / O / NR)
       - dim_rango_edad      (<18, 18-25, 26-35, 36-50, >50, NR)
       - dim_nivel_educativo (Secundaria o menos / Grado / Posgrado / NR)
       - dim_nivel_estres    (Bajo / Moderado / Alto)  <- constructo unificado
  3. Exporta un CSV "conforme" por dataset (columnas conformes + variable
     objetivo nativa) listo para cargar en MySQL o conectar a Power BI.

VALIDACION FINAL: ejecuta el filtro global de ejemplo del tutor
  ("Mujeres de 18 a 25 anos") sobre TODAS las fuentes a la vez y muestra
  el conteo por dataset: la prueba de que los silos quedaron resueltos.

USO EN WINDOWS: ajustar BASE/SALIDA como en los demas scripts del proyecto.
================================================================================
```

---
**Cómo leer este notebook.** Este proceso construye la *capa conforme* que corrige la retroalimentación de la Entrega I: mapea los atributos heterogéneos de los 12 datasets a 4 dimensiones compartidas:

| Dimensión conforme | Valores | Función |
|---|---|---|
| `dim_genero` | M / F / O / NR | Armoniza cadenas, códigos numéricos e identidades libres |
| `dim_rango_edad` | <18, 18-25, 26-35, 36-50, >50, NR | Bandas de edad; 18-25 = proxy estudiantil |
| `dim_nivel_educativo` | Secundaria o menos / Grado / Posgrado / NR | Unifica codificaciones educativas |
| `dim_nivel_estres` | Bajo / Moderado / Alto | **Constructo analítico unificado** del TFM |

Cada dataset exporta un CSV conforme (conectable a Power BI) y al final se valida el filtro global "Mujeres de 18-25" sobre todas las fuentes: la prueba de que los silos quedaron resueltos.


In [ ]:
import os
import pandas as pd

**Configuración.** Define las rutas locales (`BASE` = carpeta con los 4 bloques de datasets; `ALMC` = carpeta de entregas) y crea `capa_conforme` dentro de la entrega para los CSV de salida.

In [ ]:
# ── Configuracion global ──────────────────────────────────────────────────────
BASE = r"../data/raw"
ALMC = r"../salidas/objetivo1"

SALIDA = os.path.join(ALMC, "capa_conforme")
os.makedirs(SALIDA, exist_ok=True)   # crea la carpeta si no existe

**Funciones de armonización.** `map_genero` traduce cualquier codificación (cadenas en inglés, códigos 1/2/3 del DASS) a M/F/O/NR; `map_rango_edad` asigna la edad numérica a la banda conforme; `likert_a_nivel` aplica el criterio general de escalas 1-5 (1-2=Bajo, 3=Moderado, 4-5=Alto).

In [ ]:
# ── Mapeadores conformes ──────────────────────────────────────────────────────
def map_genero(v):
    """Armoniza cualquier codificacion de genero a M/F/O/NR."""
    if pd.isna(v):
        return "NR"
    s = str(v).strip().lower()
    if s in ("male", "m", "hombre", "1", "1.0"):
        return "M"
    if s in ("female", "f", "mujer", "2", "2.0"):
        return "F"
    if s in ("", "nan", "prefer not to say"):
        return "NR"
    return "O"   # cualquier otra identidad declarada

def map_rango_edad(v):
    """Armoniza la edad numerica a los rangos conformes."""
    try:
        e = float(v)
    except (TypeError, ValueError):
        return "NR"
    if pd.isna(e) or e < 13 or e > 90:
        return "NR"
    if e < 18:  return "<18"
    if e <= 25: return "18-25"
    if e <= 35: return "26-35"
    if e <= 50: return "36-50"
    return ">50"

def likert_a_nivel(v):
    """Escalas 1-5: 1-2=Bajo, 3=Moderado, 4-5=Alto."""
    try:
        x = float(v)
    except (TypeError, ValueError):
        return "No aplica"
    if x <= 2: return "Bajo"
    if x <= 3: return "Moderado"
    return "Alto"

In [ ]:
# ── Procesamiento por dataset ────────────────────────────────────────────────
conformes = {}   # codigo -> DataFrame conforme

**B1.1 Academic Stress.** La fuente no trae género ni edad (→ NR). La etapa académica se aprovecha como nivel educativo y el índice de estrés 1-5 se armoniza con el criterio Likert.

In [ ]:
# B1.1 Academic Stress (sin genero/edad en la fuente)
df = pd.read_csv(f"{BASE}/Bloque 1/1 Student Academic Stress Real World Dataset/"
                 "academic Stress level - maintainance 1.csv")
df.columns = [c.strip() for c in df.columns]
conformes["B1_ACADSTRESS"] = pd.DataFrame({
    "genero": "NR", "rango_edad": "NR", "nivel_educativo":
        df["Your Academic Stage"].map({"undergraduate": "Grado universitario",
                                        "post-graduate": "Posgrado",
                                        "high school": "Secundaria o menos"}).fillna("NR"),
    "nivel_estres": df["Rate your academic stress index"].map(likert_a_nivel),
    "objetivo_nativo": df["Rate your academic stress index"],
})

**B1.2 Anxiety (GAD-7).** Aplica los umbrales clínicos del GAD-7 (0-4 mínima=Bajo, 5-9 leve=Moderado, ≥10=Alto) y mapea los grados académicos al nivel educativo conforme.

In [ ]:
# B1.2 Anxiety GAD-7
df = pd.read_csv(f"{BASE}/Bloque 1/1 Student anxiety dataset/anxiety.csv",
                 encoding="latin1")
def gad_a_nivel(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x <= 4: return "Bajo"
    if x <= 9: return "Moderado"
    return "Alto"
edu_map_anx = {"Bachelor\xa0(or equivalent)": "Grado universitario",
               "Master\xa0(or equivalent)": "Posgrado",
               "Ph.D., Psy. D., MD\xa0(or equivalent)": "Posgrado",
               "High school diploma (or equivalent)": "Secundaria o menos",
               "None": "Secundaria o menos"}
conformes["B1_ANXIETY"] = pd.DataFrame({
    "genero": df["Gender"].map(map_genero),
    "rango_edad": df["Age"].map(map_rango_edad),
    "nivel_educativo": df["Degree"].map(edu_map_anx).fillna("NR"),
    "nivel_estres": df["GAD_T"].map(gad_a_nivel),
    "objetivo_nativo": df["GAD_T"],
})

**B1.3 Mental Health Survey.** Género, edad y nivel de estudios directos de la fuente; la depresión Likert 1-5 se armoniza con el criterio general.

In [ ]:
# B1.3 Mental Health Survey
df = pd.read_csv(f"{BASE}/Bloque 1/1 Student Mental Health Survey/MentalHealthSurvey.csv")
conformes["B1_MHSURVEY"] = pd.DataFrame({
    "genero": df["gender"].map(map_genero),
    "rango_edad": df["age"].map(map_rango_edad),
    "nivel_educativo": df["degree_level"].map(
        {"Undergraduate": "Grado universitario", "Postgraduate": "Posgrado"}).fillna("NR"),
    "nivel_estres": df["depression"].map(likert_a_nivel),
    "objetivo_nativo": df["depression"],
})

**B1.4 Student Habits (sintético).** `mental_health_rating` está invertida (más alto = mejor salud): 8-10=Bajo, 5-7=Moderado, 1-4=Alto. Solo alimenta la capa descriptiva.

In [ ]:
# B1.4 Student Habits (sintetico: solo capa descriptiva)
df = pd.read_csv(f"{BASE}/Bloque 1/1 Student-Habits-vs-Academic-Performance/"
                 "student_habits_performance.csv")
def salud_invertida_a_nivel(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x >= 8: return "Bajo"
    if x >= 5: return "Moderado"
    return "Alto"
conformes["B1_HABITS"] = pd.DataFrame({
    "genero": df["gender"].map(map_genero),
    "rango_edad": df["age"].map(map_rango_edad),
    "nivel_educativo": "NR",
    "nivel_estres": df["mental_health_rating"].map(salud_invertida_a_nivel),
    "objetivo_nativo": df["mental_health_rating"],
})

**B2.1 Social Media/Sleep (sintético).** Sin columna de género en la fuente (→ NR); el nivel Low/Medium/High se mapea directo. Solo capa descriptiva.

In [ ]:
# B2.1 Social Media / Sleep (sintetico)
df = pd.read_csv(f"{BASE}/Bloque 2/2 Social Media Usage vs Sleep, Stress & Productivity/"
                 "social_media_sleep_stress_productivity_11000.csv")
conformes["B2_SOCIALMEDIA"] = pd.DataFrame({
    "genero": "NR",   # la fuente no incluye genero
    "rango_edad": df["age"].map(map_rango_edad),
    "nivel_educativo": "NR",
    "nivel_estres": df["stress_level"].map({"Low": "Bajo", "Medium": "Moderado",
                                             "High": "Alto"}),
    "objetivo_nativo": df["stress_level"],
})

**B2.2 SMMH.** Se accede a las columnas por posición (los nombres son preguntas largas); la variable 'sentirse deprimido/a' 1-5 se armoniza con el criterio Likert.

In [ ]:
# B2.2 SMMH
df = pd.read_csv(f"{BASE}/Bloque 2/2 Social Media and Mental Health/smmh.csv")
cols = list(df.columns)
conformes["B2_SMMH"] = pd.DataFrame({
    "genero": df[cols[2]].map(map_genero),
    "rango_edad": df[cols[1]].map(map_rango_edad),
    "nivel_educativo": "NR",
    "nivel_estres": pd.to_numeric(df[cols[18]], errors="coerce").map(likert_a_nivel),
    "objetivo_nativo": df[cols[18]],
})

**B2.3 Lifestyle (sintético).** Mapeo directo Low/Moderate/High. Solo capa descriptiva.

In [ ]:
# B2.3 Lifestyle (sintetico)
df = pd.read_csv(f"{BASE}/Bloque 2/2 Mental_Health_Lifestyle GITHUB/"
                 "Mental_Health_Lifestyle_Dataset.csv")
conformes["B2_LIFESTYLE"] = pd.DataFrame({
    "genero": df["Gender"].map(map_genero),
    "rango_edad": df["Age"].map(map_rango_edad),
    "nivel_educativo": "NR",
    "nivel_estres": df["Stress Level"].map({"Low": "Bajo", "Moderate": "Moderado",
                                             "High": "Alto"}),
    "objetivo_nativo": df["Stress Level"],
})

**B3.1 Dreaddit (train y test).** Corpus anónimo (demografía → NR). Aplica la regla de calidad R7: excluye los fragmentos cuyo texto es un artefacto de Excel ('#NAME?'). La etiqueta binaria se mapea 0=Bajo, 1=Alto.

In [ ]:
# B3.1 Dreaddit train + test (corpus anonimo)
for part, code in [("train", "B3_DREADDIT_TR"), ("test", "B3_DREADDIT_TE")]:
    df = pd.read_csv(f"{BASE}/Bloque 3/3 Stress Analysis in Social Media Dataset/"
                     f"dreaddit-{part}.csv")
    # Regla R7: excluir artefactos de Excel del corpus
    df = df[~df["text"].str.strip().isin(["#NAME?", "#REF!", "#VALUE!"])]
    conformes[code] = pd.DataFrame({
        "genero": "NR", "rango_edad": "NR", "nivel_educativo": "NR",
        "nivel_estres": df["label"].map({0: "Bajo", 1: "Alto"}),
        "objetivo_nativo": df["label"],
    })

**B3.2 Depression Student.** Género y edad directos; la depresión Yes/No se mapea a Alto/Bajo.

In [ ]:
# B3.2 Depression Student
df = pd.read_csv(f"{BASE}/Bloque 3/3 Student_Mental_Health_Analysis GITHUB/"
                 "Depression+Student+Dataset.csv")
conformes["B3_DEPSTUDENT"] = pd.DataFrame({
    "genero": df["Gender"].map(map_genero),
    "rango_edad": df["Age"].map(map_rango_edad),
    "nivel_educativo": "NR",
    "nivel_estres": df["Depression"].map({"No": "Bajo", "Yes": "Alto"}),
    "objetivo_nativo": df["Depression"],
})

**B4.1 DASS-42.** Filtra edades válidas, calcula la subescala de estrés (clave oficial del codebook) y aplica los baremos DASS: 14-28=Bajo (normal), 29-39=Moderado, 40-56=Alto (severo+). La educación usa los códigos del codebook.

In [ ]:
# B4.1 DASS-42
df = pd.read_csv(f"{BASE}/Bloque 4/4 Depression Anxiety Stress Scales Responses/data.csv",
                 sep="\t", low_memory=False)
df = df[(df["age"] >= 13) & (df["age"] <= 90)]
str_items = [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]
df["score_estres"] = df[[f"Q{i}A" for i in str_items]].sum(axis=1)
def dass_a_nivel(v):
    # raw 14-56; equivalencia baremos DASS (raw-14 = escala estandar 0-42)
    if v <= 28: return "Bajo"        # normal
    if v <= 39: return "Moderado"    # leve / moderado
    return "Alto"                    # severo / extremadamente severo
conformes["B4_DASS"] = pd.DataFrame({
    "genero": df["gender"].map(map_genero),
    "rango_edad": df["age"].map(map_rango_edad),
    "nivel_educativo": df["education"].map({1: "Secundaria o menos",
                                             2: "Secundaria o menos",
                                             3: "Grado universitario",
                                             4: "Posgrado"}).fillna("NR"),
    "nivel_estres": df["score_estres"].map(dass_a_nivel),
    "objetivo_nativo": df["score_estres"],
})

**B4.2 Mental Health (chanchal24).** La fuente no trae edad (→ NR). El estrés creciente Yes/Maybe/No se mapea a Alto/Moderado/Bajo.

In [ ]:
# B4.2 Mental Health dataset (sin edad en la fuente)
df = pd.read_csv(f"{BASE}/Bloque 4/4 Mental Health Dataset/Mental Health dataset1.csv")
conformes["B4_MENTALHEALTH"] = pd.DataFrame({
    "genero": df["Gender"].map(map_genero),
    "rango_edad": "NR",
    "nivel_educativo": "NR",
    "nivel_estres": df["IncreasingStress"].map({"No": "Bajo", "Maybe": "Moderado",
                                                 "Yes": "Alto"}).fillna("No aplica"),
    "objetivo_nativo": df["IncreasingStress"],
})

**Exportación y validación.** Guarda un CSV conforme por dataset en `capa_conforme/` y ejecuta la consulta global de la retroalimentación ('Mujeres de 18-25') sobre todas las fuentes, mostrando el conteo por dataset y la distribución global del constructo unificado.

In [ ]:
# ── Exportacion y validacion ─────────────────────────────────────────────────
print("=" * 88)
print("EXPORTACION DE LA CAPA CONFORME")
print("=" * 88)
for code, dfc in conformes.items():
    ruta = os.path.join(SALIDA, f"conforme_{code}.csv")
    dfc.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"  {code:<18} {len(dfc):>7} filas -> {ruta}")

print()
print("=" * 88)
print("VALIDACION: filtro global del tutor -> 'Mujeres de 18 a 25 anos'")
print("(consulta imposible en el modelo v1; ahora se resuelve en una linea por fuente)")
print("=" * 88)
print(f"{'Dataset':<18} {'Total':>8} {'Mujeres 18-25':>14} {'de ellas, estres Alto':>22}")
print("-" * 66)
for code, dfc in conformes.items():
    sub = dfc[(dfc["genero"] == "F") & (dfc["rango_edad"] == "18-25")]
    alto = (sub["nivel_estres"] == "Alto").sum()
    nota = "  (sin demografia en la fuente)" if len(sub) == 0 and dfc["genero"].eq("NR").all() else ""
    print(f"{code:<18} {len(dfc):>8} {len(sub):>14} {alto:>22}{nota}")

print()
print("Distribucion global del constructo unificado (dim_nivel_estres):")
total = pd.concat([d[["nivel_estres"]] for d in conformes.values()])
print(total["nivel_estres"].value_counts().to_string())